# 02: from m/z peaks to lipid names

In notebook 01 you saw the physics. a laser fires at a tiny spot of brain, the matrix
flings the molecules into the gas phase as charged ions, and the mass spectrometer reports,
for that spot, a list of mass-to-charge values with their intensities. stack those lists
over the whole section and every pixel carries a spectrum. so far the machine has handed us
**numbers**, masses like 518.2643, and nothing else. it does not know that this number is a
lipid, let alone *which* lipid.

this notebook closes that gap. we turn a list of bare **m/z peaks** into a list of named
lipids, the operation called **annotation**. by the end you will have annotated the real raw
ions you loaded in notebook 01, and you will know exactly how much to trust each name.

we work on the genuine output of notebook 01, `data/derived/01_raw.h5ad`, the raw METASPACE
ions with their formulas and the provided Allen coordinates. nothing here is pre-cooked. we
run the same annotation the pipeline runs, save `data/derived/02_annotated.h5ad`, and hand it
to notebook 03. that is the real workflow, one stage feeding the next.

the plan, each idea built from the ground up:

> a peak is not a lipid -> adducts (one lipid, several masses) -> de-ionization -> the ppm
> formula -> matching against an LC-MS reference -> isobaric ties and how they get broken ->
> the database layer -> lipid nomenclature -> annotating every ion and saving the stage.

we keep nothing as a black box. first we unroll the mechanics in a few transparent lines so
you see exactly what happens, then we call the matching helper in `cajal_lipidomics` and
confirm it does the same thing.

In [ ]:
# the scientific-Python stack
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

# the course package: tested helpers we lean on instead of reinventing
import cajal_lipidomics as cl
from cajal_lipidomics import annotation as cann   # the ppm matcher and the ppm plot
from cajal_lipidomics import plotting as cplot     # the spectrum and spatial-map figures
from cajal_lipidomics.style import set_style, FS
set_style()   # the clean, dense, Illustrator-editable figure style of the course

# one global seed so every number and figure below is reproducible
rng = np.random.default_rng(0)

# where the data and the annotation references live
DATA = "../../data"
REFS = f"{DATA}/refs/csv"
DERIVED = f"{DATA}/derived"

print("ready: pandas", pd.__version__, "| anndata", ad.__version__)

check: you should see `ready: pandas ... | anndata ...` and no red error. a
`ModuleNotFoundError` means the notebook is on the wrong kernel. pick the `cajal-lipidomics`
kernel (top-right kernel picker) and rerun.

## 1. a peak is not a lipid

let us look at what the machine actually gave us. we load the output of notebook 01, the raw
substrate: every pixel of the two coronal sections, control (a naive female) and pregnant,
cut at the same plane, paired with the list of ions METASPACE detected. these are the raw
peaks, not yet trustworthy lipid identities.

In [ ]:
# load the raw stage from notebook 01: pixels x raw ions
raw = ad.read_h5ad(f"{DERIVED}/01_raw.h5ad")
print("raw substrate:", raw.shape, "(pixels x ions)")
print("conditions:", dict(raw.obs["Condition"].value_counts()))

# each ion carries what METASPACE reported: a molecular formula, an adduct, and an m/z
print("\nwhat the machine handed us, one row per ion:")
print(raw.var.head(8).to_string())

so a column of this matrix is not "PC 38:6", it is an ion keyed by formula and adduct,
with a measured **m/z**. METASPACE has already proposed a molecular formula for each peak (we
will use that later), but a formula is not yet a lipid name, and one formula can still hide
several different lipids. our job is to turn each m/z into a name we can defend.

let us first see the peaks as a spectrum. a spectrum is two aligned arrays: masses on one
axis, intensities on the other. we average each ion's intensity over all pixels, place each
stick at its m/z, and draw the **mean MALDI spectrum** of the dataset. every stick is one
peak the machine reported.

🔬 **TASK.** run the cell. the helper `cplot.spectrum` draws exactly the
`vlines(m/z, 0, intensity)` stick plot that mass spectrometrists call a centroid spectrum.

In [ ]:
# mean intensity per ion across all pixels -> one stick height per peak
# 🔬 your code here


stare at that plot. every stick is a number the machine is sure about: it measured
that mass to several decimal places. but not one stick is labelled with a molecule. the
instrument resolves **mass**, never **identity**. that is the central fact of this notebook: a
peak is a mass, and a mass is not a lipid.

In [ ]:
# zoom into a crowded window to show peaks crowding near the same mass
lo, hi = 505, 535
win = (mz_vals >= lo) & (mz_vals <= hi)
ax = cplot.spectrum(mz_vals[win], mean_int[win], lw=1.6,
                    title=f"zoom {lo}-{hi} Da: peaks crowd, names still unknown")
for m, h in zip(mz_vals[win], mean_int[win]):
    ax.annotate(f"{m:.3f}", (m, h), rotation=90, ha="center", va="bottom", fontsize=FS["xs"])
plt.tight_layout(); plt.show()

❓ **QUESTION.** some of those peaks sit within a few hundredths of a Dalton of each
other. if the machine had been a little less precise, could you have told them apart from the
mass alone? hold that thought. it is the whole reason annotation needs a tolerance, and the
reason ties happen.

## 2. adducts: one lipid wears several masses

here is the first twist that makes annotation non-trivial. the mass spectrometer never sees a
neutral molecule. it can only detect **ions**, molecules carrying a charge. in positive-mode
MALDI a lipid grabs a small positive ion before it flies, and the mass the machine reports is
the lipid's neutral mass **plus** the mass of whatever it grabbed. the thing it grabs is
called an **adduct**.

in positive mode brain lipids show up mostly as four adducts:

| adduct | what attaches | added mass (Da) |
|---|---|---|
| `[M+H]+`   | a proton            | 1.007276 |
| `[M+Na]+`  | a sodium ion        | 22.989769 |
| `[M+K]+`   | a potassium ion     | 38.963707 |
| `[M+NH4]+` | an ammonium ion     | 18.033823 |

so **one lipid appears at several different m/z**, one per adduct it forms. the same molecule,
seen several times, the masses spaced apart by exactly those numbers. these masses are not
arbitrary: a proton is one hydrogen nucleus, sodium and potassium are the abundant salts of
tissue, ammonium comes from the buffer. the helper stores them for us, and our raw ions
already record which adduct METASPACE assigned to each peak.

In [ ]:
# the positive-mode adduct masses the helper uses (Da added to the neutral mass)
print("ADDUCTS (positive mode):")
for name, mass in cann.ADDUCTS.items():
    print(f"  [M+{name[:-1]}]+   adds {mass:8.4f} Da")

# which adducts did our raw ions actually use?
print("\nadducts present in the raw substrate:")
print(raw.var["adduct"].value_counts().to_string())

so our dataset is dominated by potassium adducts, with smaller proton and sodium
populations. that is a real property of this matrix and tissue, potassium is abundant in
brain, so `[M+K]+` is the loudest adduct here.

let us make the idea concrete with one lipid. take PC 34:1, a common membrane
phosphatidylcholine, neutral mass 759.5778 Da. its four positive adducts land at four
different observed masses. we compute them by hand: **observed = neutral + adduct**.

In [ ]:
# PC 34:1 neutral exact mass (Da); the four adducts spread it across four observed m/z
neutral_pc34_1 = 759.5778
print(f"PC 34:1 neutral mass = {neutral_pc34_1:.4f} Da")
print("appears in the spectrum at:")
for name, shift in cann.ADDUCTS.items():
    print(f"  [M+{name[:-1]}]+  ->  m/z {neutral_pc34_1 + shift:9.4f}")

check: the same single lipid is detected at four masses spanning roughly 760 to 798 Da.
the sodium form sits 22 Da above the proton form, the potassium form 38 Da above. if you did
not know about adducts you would call these four peaks four different molecules. this is why a
raw peak list always has more peaks than lipids: adducts inflate the count, and part of
annotation is collapsing the duplicates back to one lipid.

## 3. de-ionization: subtract the adduct to recover the neutral mass

reference databases store the **neutral** exact mass of each lipid, the mass with no charge
attached, because that is the molecule's intrinsic property. the machine, by contrast, reports
the **ionized** mass. to compare the two we have to undo the ionization. the arrow of section 2
ran neutral to observed by adding an adduct, we now run it backwards:

> neutral = observed − adduct

this is **de-ionization**. for each candidate adduct we strip its mass off the observed peak
and ask: does the leftover neutral mass match a known lipid? when we do not know which adduct
formed we try all four and keep whichever gives the closest match. METASPACE already told us
the adduct for each ion, but seeing the four hypotheses side by side shows why the right one
matters.

🔬 **TASK.** take the observed peak we will spend the rest of the notebook on, m/z 518.2643
from the substrate, and de-ionize it under each of the four adduct hypotheses.

In [ ]:
# the real MSI peak we will annotate, straight from the raw substrate
# 🔬 your code here


so the same observed peak corresponds to four *different* neutral masses, one per adduct
guess. the right guess is the one whose neutral mass matches a real lipid in a reference list.
to decide what "matches" means we need a tolerance, and the honest way to write a mass
tolerance is parts per million.

## 4. the ppm formula: how close is close enough

no two real numbers are ever exactly equal, and the machine's mass has tiny error. so a match
is "the observed mass and the reference mass agree **to within a small tolerance**". we do not
measure that tolerance in absolute Daltons, because the same absolute error means something
different at mass 200 than at mass 1000. we measure it in **parts per million**:

> ppm error = 1 000 000 × |observed − reference| / reference

it is the fractional disagreement, scaled up by a million so the numbers are readable. the
course uses a **5 ppm** window, the same threshold as the paper. crucially, a fixed ppm window
is a *proportional* window in Daltons: 5 ppm is a wider Dalton gap at high mass than at low
mass.

🔬 **TASK.** the helper `cann.ppm_error` is this one formula. use it to confirm the headline
number: **5 ppm at m/z 800 is a 0.004 Da window**.

💡 **HINT: read the code you run.** every helper in this notebook lives in one short, readable file, `src/cajal_lipidomics/annotation.py`. open it now and find `ppm_error`. it is the single line `1e6 * np.abs(observed - reference) / reference`, exactly the formula above. you will trust the number more once you have seen there is no magic behind it.

In [ ]:
# the ppm formula, as one helper call; it is literally 1e6 * |obs - ref| / ref
# 🔬 your code here


check: at m/z 800, 5 ppm is 0.004 Da, four thousandths of a Dalton. that is
astonishingly tight: high-resolution MS measures mass so precisely that a 4 mDa window already
separates most lipids. but not all. some lipids share a mass closer than 4 mDa, and those
collide inside the window. we will meet exactly such a collision in a moment.

💡 **HINT.** read the formula as a sentence. "how far apart are these two masses, as a fraction
of the mass, times a million?" if two masses agree to 1 part in a million the ppm error is 1,
our cutoff of 5 means "agree to within 5 parts per million".

## 5. matching against an LC-MS reference

now we need a reference list of lipids whose masses we trust, to match our peaks against. the
strongest reference is an orthogonal measurement: **liquid chromatography coupled to mass
spectrometry (LC-MS)** run on similar brain tissue. LC-MS separates lipids in time before
weighing them, and a tandem step (LC-MS/MS) fragments each lipid to read its structure. the
result is a list of **confidently identified lipids with their exact ionized masses**. if a
confirmed LC-MS lipid sits within 5 ppm of one of our MSI peaks, we borrow its name. that is
the core of annotation.

we load the in-house LC-MS reference. the helper `cann.load_lcms_reference` reads the file and
keeps three columns: the lipid name, its adduct, and the observed m/z (already ionized, so we
compare it directly to MSI peaks).

In [ ]:
# the in-house LC-MS reference: confidently identified lipids with ionized m/z
lcms = cann.load_lcms_reference(f"{REFS}/lcms_mar2022_withcounterions (2).txt")
print("LC-MS reference:", lcms.shape, "(lipid, adduct, m/z) rows")
print(lcms.head(8).to_string())
print("\nthe same lipid appears under several adducts (h, na, k, nh4),",
      "exactly the adduct spread from section 2")

before we call the helper, let us unroll the match by hand so there is no magic. for our
observed peak we compute the ppm error to **every** reference peak, then keep the ones inside
the 5 ppm window. this is the whole matcher in three lines.

In [ ]:
# UNROLL the matcher: ppm error of every reference peak to our observed peak
ppm_to_all = cann.ppm_error(observed, lcms["m/z"].to_numpy())
inside = ppm_to_all <= 5.0
hits_manual = lcms.loc[inside].assign(ppm=ppm_to_all[inside]).sort_values("ppm")

print(f"observed peak m/z {observed:.4f}")
print(f"reference peaks within 5 ppm: {int(inside.sum())}")
print(hits_manual.to_string(index=False))

the helper `cann.match_lcms` does exactly this: compute `ppm_error` to every reference
peak, keep those within the tolerance, sort by ppm distance. we confirm it returns the same
rows.

💡 **HINT: read the code you run.** open `src/cajal_lipidomics/annotation.py` again and read `match_lcms`. it is the same three lines you just wrote by hand: call `ppm_error` against every reference m/z, keep the ones within `ppm_tol`, sort by ppm distance. while you are there, glance at `load_lcms_reference` so you know which columns it keeps.

In [ ]:
# the helper does exactly the three lines above
hits = cann.match_lcms(observed, lcms, ppm_tol=5.0)
print(hits.to_string(index=False))
assert set(hits["Lipid"]) == set(hits_manual["Lipid"]), "helper and manual must agree"
print("\nhelper and hand-rolled matcher agree.")

check: two reference lipids fall inside the 5 ppm window: **LPC 15:1 as [K]** and
**LPE 18:1 as [K]**, both at m/z 518.2644, both a hundredth of a ppm from our peak. notice the
adduct: both are the **potassium** forms, matching the `[M+K]+` that METASPACE assigned. so our
de-ionization guess from section 3 was the potassium hypothesis, and subtracting 38.96 Da gives
the shared neutral mass these two lipids happen to have.

this is the moment the notebook has been building toward. the window did its job, but it
returned **two** lipids, not one. we have a tie.

## 6. making the match visible: the LC-MS ↔ MSI ppm plot

a table of ppm distances is correct but cold. the course ships a figure that makes the matching
operation obvious at a glance. it stacks two stick spectra on a shared m/z axis: on top the
LC-MS reference peaks near our target, on the bottom the single MSI peak we are annotating. a
shaded gold band marks the ±5 ppm acceptance window. reference sticks inside the band are
accepted (green), outside are rejected (gray), and each is labelled with its name, adduct, and
ppm distance.

🔬 **TASK.** run `cann.plot_ppm_match` on our peak. read it like this: the dashed line is the
observed peak, the gold band is the 5 ppm tolerance, and any green label is a lipid that
donates its name.

💡 **HINT: read the code you run.** before you call `plot_ppm_match`, open `src/cajal_lipidomics/annotation.py` and read it top to bottom. it is plain matplotlib: it shades the ±ppm band, draws the reference sticks on top and the single MSI peak below, and colors a stick green when its ppm distance falls inside the window. nothing it draws is hidden from you.

In [ ]:
# the side-by-side ppm plot: LC-MS reference (top) vs the MSI peak (bottom)
# 🔬 your code here


that single figure carries the whole lesson of sections 2 to 5. adducts spread the
reference lipids across the axis. the ±5 ppm band is tiny but real. and two lipids, LPC 15:1
and LPE 18:1, both land their potassium adduct exactly on our peak. they are **isobaric**:
different molecules with the same mass, indistinguishable by mass alone. the ppm window cannot
break this tie, because the tie is at essentially 0 ppm. we need outside information.

❓ **QUESTION.** look at the gray sticks outside the gold band. they are real lipids in the
reference, but they sit too far from our peak. in your own words, why do we reject them even
though they are "close"?

## 7. breaking the isobaric tie, and what our pipeline actually does

why do LPC 15:1 and LPE 18:1 share a mass? because they share a molecular formula. both are
C23H46NO7P. swapping a choline head for an ethanolamine head and trading three acyl carbons for
the matching number of hydrogens lands you on the identical atom count, and identical atoms
mean identical exact mass. mass cannot tell them apart, and neither can METASPACE's formula,
which is why our raw `var` carries the formula C23H46NO7P with no further commitment.

In [ ]:
# what the automatic pipeline picks: the top-ranked hit by ppm distance
top = hits.iloc[0]
print("our automatic annotation for this peak:")
print(f"  {top['Lipid']}  [{top['Adduct']}]  at {top['ppm']:.3f} ppm")
print("\nthe rival candidate, tied at the same mass:")
print(f"  {hits.iloc[1]['Lipid']}  [{hits.iloc[1]['Adduct']}]  at {hits.iloc[1]['ppm']:.3f} ppm")

so our automatic call is **LPC 15:1**. but is that biologically the right molecule? mass
cannot say, yet **biology** can. in brain tissue these two are not equally plausible. the
tie-breaker is a **quantitative bulk LC-MS** measurement: a separate run that reports *how much*
of each lipid is in brain, in nmol percent of total lipid. if one candidate is abundant and the
other essentially absent, the abundant one almost certainly made our peak. we load
`QuantitativeLCMS.csv` and look up our two candidates.

In [ ]:
# bulk quantitative LC-MS: how much of each lipid is in brain (nmol % of total lipid)
qlcms = pd.read_csv(f"{REFS}/QuantitativeLCMS.csv").dropna(subset=["LIPID_ID"]).copy()
female_cols = ["Female", "Female.1", "Female.2", "Female.3"]
for c in female_cols:
    qlcms[c] = pd.to_numeric(qlcms[c], errors="coerce")
qlcms["abundance"] = qlcms[female_cols].mean(axis=1)   # mean over the 4 female samples

for cand in ["LPE 18:1", "LPC 15:1"]:
    row = qlcms.loc[qlcms["LIPID_ID"] == cand, "abundance"]
    if len(row):
        print(f"  {cand:10s}: {row.iloc[0]:.5f} nmol% of total lipid  (MEASURED in bulk brain)")
    else:
        print(f"  {cand:10s}: not detected in the bulk LC-MS reference at all")

check: LPE 18:1 is measured in bulk brain at 0.0199 nmol percent. LPC 15:1 is **not
detected at all**, not as a typo but because **odd-chain lyso-phosphatidylcholines are
vanishingly rare in mammalian brain**: animal fatty acids are overwhelmingly even-chain. let us
confirm there is no odd-chain LPC anywhere in the quantitative reference.

In [ ]:
# are there ANY odd-chain LPC species in the bulk reference?
lpc = qlcms[qlcms["LIPID_ID"].astype(str).str.startswith("LPC")]
odd = [s for s in lpc["LIPID_ID"] if any(f" {n}:" in str(s) for n in (13, 15, 17, 19, 21, 23))]
print("LPC species measured in bulk brain:", sorted(lpc["LIPID_ID"].tolist()))
print("\nodd-chain LPC species found:", odd if odd else "NONE")

so the honest situation is this. our **automatic** matcher, which only knows mass, calls
the peak **LPC 15:1**. a fuller pipeline that also weighs bulk abundance, the rule the paper
uses, would overrule that and call it **LPE 18:1**, the candidate that actually exists in brain.
the paper formalizes the rule: for the candidates of one peak, normalize their bulk molar
fractions and, if the top one exceeds 80 percent of the summed abundance, keep only it. here
LPE 18:1 takes 100 percent of the measurable abundance (its rival contributes zero), so it wins
outright. in the full EUCLID pipeline that step is
`Preprocessing.abundance_prioritization_lcms(matched_table, lcms_csv, threshold=0.8)`, but it
is the same 80-percent rule on the same numbers you just printed.

we keep the simple mass-only matcher in this course, so this peak will be saved as **LPC 15:1**.
that is a deliberate, honest choice. your annotations are produced by a transparent rule you can
read end to end, and where that rule is weaker than a hand-curated pipeline, you can see exactly
where and why. a few of your names will differ from the paper. that is the realistic texture of
real annotation, not a mistake to hide.

❓ **QUESTION.** suppose both candidates had been abundant in brain, say a 55/45 split. the
80-percent rule would refuse to choose. what would you report then, and why is honesty about the
ambiguity better than picking one arbitrarily?

## 8. the database layer: LIPID MAPS

the LC-MS reference is a curated, brain-specific list. but many peaks have no LC-MS hit, and for
those we fall back to **chemical databases** that store the neutral exact mass of every known
lipid structure. the one that matters here is **LIPID MAPS** (`structures.sdf`), the reference
structure database for lipids. each entry carries a structure, a formula, the **neutral exact
mass**, and a shorthand abbreviation like `PC 34:1`. (METASPACE itself annotates against a
sibling database, HMDB, the Human Metabolome Database, which is why the project ships an
HMDB↔LIPID MAPS bridge, but the matching logic is identical.)

the mechanics are exactly what you already did: de-ionize under each adduct, then ppm-match
against the database's `EXACT_MASS`. we open LIPID MAPS, read a handful of entries so you see
the fields, and run the same de-ionize-and-match logic against it for our peak.

In [ ]:
# peek at LIPID MAPS: structures with neutral exact masses and shorthand abbreviations
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")   # silence parser chatter

# read EVERY structure in the database, no shortcut: we match against the whole of LIPID MAPS
supp = Chem.SDMolSupplier(f"{DATA}/refs/structures.sdf")
records = []
for mol in supp:
    if mol is None:
        continue
    p = mol.GetPropsAsDict()
    if "EXACT_MASS" in p and "ABBREVIATION" in p:
        records.append((p["ABBREVIATION"], float(p["EXACT_MASS"]), p.get("FORMULA", "")))
lipidmaps = pd.DataFrame(records, columns=["ABBREVIATION", "EXACT_MASS", "FORMULA"])
print(f"parsed {len(lipidmaps)} LIPID MAPS structures (the full database)")
print(lipidmaps.head(5).to_string(index=False))

In [ ]:
# de-ionize our peak under each adduct, find the closest LIPID MAPS neutral mass within 5 ppm
masses = lipidmaps["EXACT_MASS"].to_numpy()
names = lipidmaps["ABBREVIATION"].to_numpy()
best_name, best_ppm, best_adduct = None, np.inf, None
for adduct, shift in cann.ADDUCTS.items():
    neutral = observed - shift                 # de-ionize
    j = int(np.argmin(np.abs(masses - neutral)))
    e = cann.ppm_error(neutral, masses[j])     # the same ppm formula
    if e <= 5.0 and e < best_ppm:
        best_name, best_ppm, best_adduct = names[j], e, adduct
print(f"closest LIPID MAPS hit for m/z {observed:.4f}:")
print(f"  {best_name}  as [M+{best_adduct[:-1]}]+   ({best_ppm:.2f} ppm)" if best_name
      else "  no hit within 5 ppm")

check: LIPID MAPS, with no abundance information at all, lands on **LPC 15:1 under the
potassium adduct at about 1 ppm**, one of the two isobars from our tie. that is the key lesson
about the database layer: LIPID MAPS stores *every* known lipid structure, including the
biologically rare LPC 15:1, so the database can confirm a mass exists but it **cannot break the
isobaric tie either**. both LPC 15:1 and LPE 18:1 are in it as legitimate structures. only the
*quantitative* bulk LC-MS from section 7 knows what is actually abundant in brain. the databases
give breadth (every known structure), LC-MS gives confidence (this lipid was really seen, and
quantified, in brain).

💡 **HINT.** the full annotator is one call, `Preprocessing.annotate_molecules(...)`, which runs
precisely the de-ionize-and-ppm-match loop you wrote above against LIPID MAPS, then merges the
LC-MS reference layer on top. you have now built its core by hand, so the one-liner holds no
mystery.

## putting it together: how the full pipeline scores trust

we matched against one curated LC-MS list, but the paper matches each peak against several
references at once: two ESI LC-MS runs, an LC-MS/MS run, METASPACE (HMDB), LIPID MAPS, and two
published brain datasets. each reference that agrees with a peak adds a confidence weight, and the
weight says how much that source is worth. structural LC-MS/MS identification counts 8, ESI LC-MS
counts 2, a published study counts 1, METASPACE counts 1 when its FDR is below 0.05 and 0.5
otherwise.

## 9. annotate every ion: m/z columns become lipid names

we have done one peak thoroughly. now we run the same matcher over **all** the raw ions and
write a lipid name onto each. this is exactly what the pipeline's `annotate` step does: for
every ion, take the top LC-MS hit within 5 ppm, tidy the name, and fall back to a placeholder
`ion_<mz>` when nothing matches. peaks that match get a real name, peaks that do not stay flagged
as unannotated.

🔬 **TASK.** run the loop. it is the matcher from section 5, applied 104 times.

In [ ]:
# annotate every raw ion: top LC-MS hit within 5 ppm, else an ion_<mz> placeholder
# 🔬 your code here


check: about 63 of the 104 ions earn a confident lipid name, and roughly a third of the
matched ones carried an isobaric tie that the top-hit rule silently resolved (the same kind of
tie you dissected for LPC 15:1 vs LPE 18:1). the rest stay as `ion_<mz>` placeholders. that
ratio is honest: not every peak a mass spectrometer reports is a confidently identifiable lipid,
and pretending otherwise would be the dishonest move.

let us look at the lipid classes our annotation recovered.

In [ ]:
# how many lipids per class did we annotate?
cls = raw.var.loc[annotated, "lipid"].str.split(r"[ (]").str[0]
print("annotated lipids per class:")
print(cls.value_counts().to_string())

the broad strokes of brain lipid chemistry are already visible: glycerophospholipids
(PC, PE, PG, PA, PS) dominate, with sphingolipids (SM, HexCer) and a handful of lyso-species.
exactly the biology we expect from a coronal brain section.

## 10. reading a lipid name: nomenclature → properties

a name like `PC 38:6` is itself a compressed sentence, and to group and color lipids later we
need to unpack it into properties. the shorthand encodes three things:

> `PC 38:6` = class **PC** (phosphatidylcholine), **38** total acyl carbons across the tails,
> **6** carbon-carbon double bonds (the degree of unsaturation).

because MALDI cannot tell isomers apart, these are **sum compositions**: 38 total carbons, not
"an 18 and a 20". two wrinkles appear in real names: **ether lipids** mark an ether-linked chain
with `O-` or `P-`, as in `PE O-38:7`, and **sphingolipids** can spell the sphingoid base and the
N-acyl chain separately, like `SM 18:1;O2/24:1`. we extract class, carbons, and double bonds
with a few regular expressions, the same approach the paper used to build its lipid property
table.

In [ ]:
# a transparent regex parser: class, total carbons, double bonds, ether flag
def parse_lipid(name):
    lipid_class = re.split(r"[ (]", name)[0]          # text before first space/paren -> class
    ether = bool(re.search(r"\bO-|\bP-", name))      # O-/P- prefix -> ether lipid
    m = re.search(r"(\d+):(\d+)", name)              # first 'C:DB' -> sum composition
    carbons = int(m.group(1)) if m else None
    double_bonds = int(m.group(2)) if m else None
    return lipid_class, carbons, double_bonds, ether

for n in ["PC 38:6", "PE O-38:7", "SM 18:1;O2/24:1", "LPC 15:1", "LPE 18:1"]:
    c, nc, db, e = parse_lipid(n)
    print(f"  {n:18s} -> class={c:5s} carbons={nc}  double_bonds={db}  ether={e}")

# the per-chain caveat: SM 18:1;O2/24:1 has TWO chains, the naive regex sees only the first
sm = "SM 18:1;O2/24:1"
pairs = re.findall(r"(\d+):(\d+)", sm)
true_c = sum(int(c) for c, _ in pairs); true_db = sum(int(d) for _, d in pairs)
print(f"\n  caveat on {sm!r}:")
print(f"    naive regex grabs the first chain only -> {parse_lipid(sm)[1]}:{parse_lipid(sm)[2]} (the sphingoid base)")
print(f"    correct total across both chains        -> {true_c}:{true_db}  (the real sum composition)")

check: the parser reads `PC 38:6` as PC / 38 / 6, flags `PE O-38:7` as an ether lipid,
and on `SM 18:1;O2/24:1` it grabs only the first `C:DB`, the sphingoid base 18:1, not the full
molecule. that last case is a real caveat: the molecule has two chains, an 18:1 sphingoid base
and a 24:1 N-acyl tail, so the correct sum composition is 42:2. the paper handles all of these
robustly with goslin, a dedicated lipid nomenclature tool, and the project ships a tested
`compress_lipid_name_robust` for exactly this. for our sum-composition names, where each name
already carries a single total, the simple regex is enough.

now run the parser across all annotated lipids and build a property table.

In [ ]:
# parse every annotated lipid into a property table
names_ann = raw.var.loc[annotated, "lipid"]
mz_ann = raw.var.loc[annotated, "mz"].to_numpy(float)
props = pd.DataFrame([parse_lipid(n) for n in names_ann],
                     columns=["class", "carbons", "double_bonds", "ether"],
                     index=names_ann.index)
props["mz"] = mz_ann
print("property table:", props.shape)
print(props.head(8).to_string())

these properties are exactly what later notebooks color and group by. a quick look: do
the classes spread across mass and unsaturation the way lipid chemistry predicts? we plot every
annotated lipid by carbons against double bonds, sized by mass, colored by class with the
course's distinct-color palette (never `tab20`).

In [ ]:
# the property landscape: carbons vs double bonds, colored by class
fig, ax = plt.subplots(figsize=(7.5, 5))
classes = props["class"].value_counts().index
palette = cplot.distinct_colors(len(classes))
for col, c in zip(palette, classes):
    sub = props[props["class"] == c]
    ax.scatter(sub["carbons"], sub["double_bonds"], s=18 + sub["mz"] / 20,
               color=col, alpha=0.8, label=c, edgecolor="none")
ax.set_xlabel("total acyl carbons"); ax.set_ylabel("double bonds (unsaturation)")
ax.set_title("the annotated lipids by class, carbons and unsaturation (size ~ m/z)")
ax.legend(ncol=3, fontsize=FS["xs"], loc="upper left")
plt.tight_layout(); plt.show()

❓ **QUESTION.** the lyso-lipids (LPC, LPE, with one tail removed) cluster at low carbon
counts, while the diacyl phospholipids spread to 40+ carbons. why does losing a fatty tail move
a lipid to the left on this plot?

## 11. a named lipid draws a picture of the brain, and we save the stage

the final proof that annotation is real: a true lipid traces anatomy, pure noise would not. we
take our worked peak, now named, and map its intensity across both sections. the helper
`cplot.spatial_lipid` places every pixel at its CCF coordinate and colors it by intensity.

In [ ]:
# build a view whose columns are lipid names, so we can ask for one by name
named = raw.copy()
named.var_names = ad.utils.make_index_unique(pd.Index(raw.var["lipid"].astype(str)))

peak_name = raw.var["lipid"].iloc[0]      # our worked peak: LPC 15:1
print(f"mapping '{peak_name}' (the ion at m/z {observed:.4f}) across both sections")
cplot.spatial_lipid(named, peak_name, section_key="SectionID", title_key="Condition")
plt.suptitle(f"{peak_name}  (m/z {observed:.4f}, finally named and mapped)",
             fontsize=FS["m"], y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# a table to eyeball every annotation we assigned
cols = [c for c in ["lipid", "mz", "ppm", "formula", "adduct"] if c in raw.var.columns]
annot_tbl = raw.var[cols].copy()
print(f"all {raw.n_vars} ions and their annotated lipid names:")
print(annot_tbl.to_string())

check: the map is not random speckle. the named lipid paints coherent anatomical
structure, and the same pattern appears in control and pregnant. that spatial coherence is the
final confirmation the annotation is real: a bare number became a named molecule, and the named
molecule draws a picture of the brain.

now we save this stage. notebook 03 will load `02_annotated.h5ad` and normalize it.

In [ ]:
# save the annotated stage: same matrix, now with lipid names in var
raw.write_h5ad(f"{DERIVED}/02_annotated.h5ad")
print("wrote", f"{DERIVED}/02_annotated.h5ad")
print("var now carries:", list(raw.var.columns))
print(f"annotated {int(annotated.sum())}/{raw.n_vars} ions; the rest stay flagged ion_<mz>")

## what you can now do

- explain why an MSI peak is a **mass, not a lipid**, until it is annotated.
- list the four positive-mode **adducts** and compute the masses one lipid shows up at.
- **de-ionize** an observed peak (neutral = observed − adduct) to compare against a database.
- write the **ppm formula** and state that 5 ppm at m/z 800 is a 0.004 Da window.
- match a peak against an **LC-MS reference** with `cann.match_lcms` and read the side-by-side
  ppm plot from `cann.plot_ppm_match`.
- recognise an **isobaric tie**, see how a top-hit rule resolves it, and how bulk LC-MS
  abundance (the 80-percent rule) would resolve it more faithfully.
- parse a lipid name into **class, carbons, double bonds, ether** with a regex.
- run the matcher over every raw ion and **save the annotated stage** for notebook 03.

🔬 **TASK (optional, for the curious).** pick another peak from the substrate, say a PC near
m/z 800, run `cann.match_lcms` and `cann.plot_ppm_match` on it, and see whether it is a clean
single hit or another isobaric tie. the more crowded the mass region, the more ties you find.

In [ ]:
# OPTIONAL: try the workflow on a peak you choose. edit `my_peak` and run.
# 🔬 your code here
